In [22]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from supabase import create_client
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics.pairwise import euclidean_distances

load_dotenv("../../.env")

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

In [23]:
# import data to pandas
trials = supabase.table("trials").select("*").eq("trial_type", "training").execute().data
shake_reports = supabase.table("shake_reports").select("*").execute().data
morphology_profiles = supabase.table("morphology_profiles").select("*").execute().data

trials_df = pd.DataFrame(trials)
shake_df = pd.DataFrame(shake_reports)
morph_df = pd.DataFrame(morphology_profiles)

display(trials_df)
display(shake_df)
display(morph_df)

,id,trial_type,collection_id,weight,morphology_profile_id,smitski_name_id
0,2,training,1,42.8,2,2
1,3,training,2,38.5,3,3
2,4,training,3,37.6,4,4
3,5,training,4,38.7,5,5
4,6,training,5,46.0,6,6
5,1,training,1,45.6,1,1
6,7,training,6,38.2,7,7


,id,trial_id,shake_position,movement_amount,loudness,sound_hardness
0,12,2,rotated_2,3,3,2.0
1,13,2,up_down,3,3,2.0
2,11,2,rotated_1,2,2,1.0
3,10,2,base_rotation,1,1,1.0
4,7,1,rotated_1,1,1,0.0
5,14,3,base_rotation,1,1,0.0
6,15,3,rotated_1,2,2,2.0
7,16,3,rotated_2,2,2,1.0
8,17,3,up_down,3,3,0.0
9,18,4,base_rotation,3,3,2.0


,id,is_symmetrical,smiski_size,total_prop_number,largest_prop_size,multi_body,elongation_profile,compactness,body_type,multi_body_is_joined,multi_body_has_smaller_body
0,1,False,standard,1,0.0,False,2,1,NaN,None,None
1,2,True,standard,0,NaN,False,1,1,NaN,None,None
2,3,False,standard,0,NaN,False,2,0,NaN,None,None
3,4,False,standard,0,NaN,False,1,1,normal,None,None
4,5,True,standard,0,NaN,False,1,2,normal,None,None
5,6,False,smaller,2,1.0,True,1,0,NaN,False,False
6,7,False,standard,0,NaN,False,1,1,NaN,None,None


In [24]:
# build ml dataframe
trial_labels_df = trials_df.merge(
    morph_df,
    left_on="morphology_profile_id",
    right_on="id",
    suffixes=("_trial", "_morph")
)

shake_wide = shake_df.pivot(
    index="trial_id",
    columns="shake_position",
    values=["movement_amount", "loudness", "sound_hardness"]
)
shake_wide.columns = [f"{position}_{feature}" for feature, position in shake_wide.columns]
shake_wide = shake_wide.reset_index()

ml_df = trial_labels_df.merge(shake_wide, left_on="id_trial", right_on="trial_id")
ml_df

,id_trial,trial_type,collection_id,weight,morphology_profile_id,smitski_name_id,id_morph,is_symmetrical,smiski_size,total_prop_number,...,rotated_2_movement_amount,up_down_movement_amount,base_rotation_loudness,rotated_1_loudness,rotated_2_loudness,up_down_loudness,base_rotation_sound_hardness,rotated_1_sound_hardness,rotated_2_sound_hardness,up_down_sound_hardness
0,2,training,1,42.8,2,2,2,True,standard,0,...,3.0,3.0,1.0,2.0,3.0,3.0,1.0,1.0,2.0,2.0
1,3,training,2,38.5,3,3,3,False,standard,0,...,2.0,3.0,1.0,2.0,2.0,3.0,0.0,2.0,1.0,0.0
2,4,training,3,37.6,4,4,4,False,standard,0,...,3.0,3.0,3.0,3.0,3.0,3.0,2.0,2.0,2.0,1.0
3,5,training,4,38.7,5,5,5,True,standard,0,...,1.0,1.0,1.0,2.0,1.0,1.0,0.0,1.0,0.0,0.0
4,6,training,5,46.0,6,6,6,False,smaller,2,...,1.0,1.0,0.0,0.0,1.0,1.0,NaN,NaN,0.0,0.0
5,1,training,1,45.6,1,1,1,False,standard,1,...,3.0,2.0,3.0,1.0,3.0,2.0,1.0,0.0,1.0,1.0
6,7,training,6,38.2,7,7,7,False,standard,0,...,1.0,2.0,3.0,2.0,1.0,2.0,2.0,1.0,0.0,1.0


In [25]:
# make rotation relevant features
rotation_positions = ["base_rotation", "rotated_1", "rotated_2", "up_down"]
metrics = ["movement_amount", "loudness", "sound_hardness"]

for metric in metrics:
    cols = [f"{pos}_{metric}" for pos in rotation_positions]
    ml_df[f"mean_{metric}"] = ml_df[cols].mean(axis=1)
    ml_df[f"variance_{metric}"] = ml_df[cols].var(axis=1)
    ml_df[f"range_{metric}"] = ml_df[cols].max(axis=1) - ml_df[cols].min(axis=1)
    rotation_mean = ml_df[[f"{pos}_{metric}" for pos in rotation_positions[:3]]].mean(axis=1)
    ml_df[f"updown_vs_rotation_{metric}"] = ml_df[f"up_down_{metric}"] - rotation_mean

feature_cols = [
    "weight",
    "mean_movement_amount", "variance_movement_amount", "range_movement_amount",
    "mean_loudness", "variance_loudness", "range_loudness",
    "mean_sound_hardness", "variance_sound_hardness", "range_sound_hardness",
    "updown_vs_rotation_movement_amount", "updown_vs_rotation_loudness", "updown_vs_rotation_sound_hardness",
]

# # alternative rotation feature method
# metrics = ["movement_amount", "loudness", "sound_hardness"]

# for metric in metrics:
#     up = ml_df[f"up_down_{metric}"]
#     base = ml_df[f"base_rotation_{metric}"]
#     r1 = ml_df[f"rotated_1_{metric}"]
#     r2 = ml_df[f"rotated_2_{metric}"]

#     # up_down vs base rotation contrast
#     ml_df[f"updown_minus_base_{metric}"] = up - base

#     # max vs min across all positions (avoid division by zero with + 0.01)
#     all_pos = pd.concat([up, base, r1, r2], axis=1)
#     ml_df[f"max_over_min_{metric}"] = all_pos.max(axis=1) / (all_pos.min(axis=1) + 0.01)

#     # horizontal (rotations) vs vertical (up_down) contrast
#     rotation_mean = (base + r1 + r2) / 3
#     ml_df[f"vertical_vs_horizontal_{metric}"] = up / (rotation_mean + 0.01)

# feature_cols.extend([
#     f"updown_minus_base_{metric}",
#     f"max_over_min_{metric}",
#     f"vertical_vs_horizontal_{metric}",
# ] for metric in metrics)

# # flatten since extend with a generator gives nested lists
# feature_cols = [f for group in [
#     [
#         f"updown_minus_base_{metric}",
#         f"max_over_min_{metric}",
#         f"vertical_vs_horizontal_{metric}",
#     ]
#     for metric in metrics
# ] for f in group] + feature_cols


In [26]:
# prepare targets
cols_to_drop = ["body_type", "multi_body_is_joined", "multi_body_has_smaller_body"]
bool_cols = ["is_symmetrical", "multi_body"]
text_cols = ["smiski_size"]

target_cols = [c for c in morph_df.columns if c != "id" and c not in cols_to_drop]

y_multi = ml_df[target_cols].copy()
y_multi["largest_prop_size"] = y_multi["largest_prop_size"].fillna(-1)

encoders = {}
for col in text_cols:
    le = LabelEncoder()
    y_multi[col] = le.fit_transform(y_multi[col].astype(str))
    encoders[col] = le

for col in bool_cols:
    y_multi[col] = y_multi[col].astype(int)

In [27]:
# train
X = ml_df[feature_cols].fillna(ml_df[feature_cols].median())

multi_model = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
multi_model.fit(X, y_multi)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,RandomForestC...ndom_state=42)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the f

In [28]:
# evaluation

y_multi_arr = y_multi.values
X_arr = X.values
per_target_correct = {col: 0 for col in target_cols}
n_splits = 0

for train_idx, test_idx in LeaveOneOut().split(X_arr):
    fold_model = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
    fold_model.fit(X_arr[train_idx], y_multi_arr[train_idx])
    fold_pred = fold_model.predict(X_arr[test_idx])
    for i, col in enumerate(target_cols):
        if fold_pred[0][i] == y_multi_arr[test_idx][0][i]:
            per_target_correct[col] += 1
    n_splits += 1

print(f"LOOCV results ({n_splits} folds):\n")
for col, correct in per_target_correct.items():
    print(f"  {col}: {correct}/{n_splits} correct ({100*correct/n_splits:.0f}%)")

LOOCV results (7 folds):

  is_symmetrical: 2/7 correct (29%)
  smiski_size: 6/7 correct (86%)
  total_prop_number: 4/7 correct (57%)
  largest_prop_size: 4/7 correct (57%)
  multi_body: 6/7 correct (86%)
  elongation_profile: 3/7 correct (43%)
  compactness: 4/7 correct (57%)


In [29]:
# fake shake data
fake_shake = pd.DataFrame([{
    "weight": 42.0,
    "mean_movement_amount": 1.5, "variance_movement_amount": 0.9, "range_movement_amount": 2.0,
    "mean_loudness": 1.5, "variance_loudness": 0.9, "range_loudness": 2.0,
    "mean_sound_hardness": 0.75, "variance_sound_hardness": 0.25, "range_sound_hardness": 1.0,
    "updown_vs_rotation_movement_amount": 0.5,
    "updown_vs_rotation_loudness": 0.3,
    "updown_vs_rotation_sound_hardness": -0.2,
    "updown_minus_base_movement_amount": 0.5,
    "updown_minus_base_loudness": 0.3,
    "updown_minus_base_sound_hardness": -0.2,
    "max_over_min_movement_amount": 2.0,
    "max_over_min_loudness": 2.0,
    "max_over_min_sound_hardness": 1.5,
    "vertical_vs_horizontal_movement_amount": 1.2,
    "vertical_vs_horizontal_loudness": 1.1,
    "vertical_vs_horizontal_sound_hardness": 0.9,
}])[feature_cols]  # reorder to match training

raw_profile = dict(zip(target_cols, multi_model.predict(fake_shake)[0]))

decoded_profile = {}
for attr, val in raw_profile.items():
    if attr in encoders:
        decoded_profile[attr] = encoders[attr].inverse_transform([int(val)])[0]
    elif attr in bool_cols:
        decoded_profile[attr] = bool(val)
    else:
        decoded_profile[attr] = val

if decoded_profile.get("total_prop_number") == 0:
    decoded_profile["largest_prop_size"] = None
else:
    decoded_profile["largest_prop_size"] = int(decoded_profile["largest_prop_size"])

print("Predicted morphology profile:")
for attr, val in decoded_profile.items():
    print(f"  {attr}: {val}")

Predicted morphology profile:
  is_symmetrical: False
  smiski_size: standard
  total_prop_number: 0.0
  largest_prop_size: None
  multi_body: False
  elongation_profile: 1.0
  compactness: 1.0


In [30]:
# simulate user selecting collection
selected_collection_id = 1  # change this to test different collections

# get the smiskis in that collection with their morphology profiles
collection_smiskis = supabase.table("smiski_names")\
    .select("*, morphology_profiles(*)")\
    .eq("collection_id", selected_collection_id)\
    .execute().data

print(f"Smiskis in collection {selected_collection_id}:")
for s in collection_smiskis:
    print(f"  {s['name']} → profile {s['morphology_profile_id']}")

Smiskis in collection 1:
  SMISKI Hoop → profile 1
  SMISKI Doing Crunches → profile 2


In [31]:
# build candidate profile matrix from this collection
candidate_cols = target_cols  # same attributes model was trained on

candidate_rows = []
candidate_names = []

for smiski in collection_smiskis:
    profile = smiski["morphology_profiles"]
    if profile is None:
        continue

    row = {}
    for col in candidate_cols:
        val = profile.get(col)
        # apply same encoding as training
        if col in encoders:
            try:
                val = encoders[col].transform([str(val)])[0]
            except:
                val = 0
        elif col in bool_cols:
            val = int(bool(val)) if val is not None else 0
        elif col == "largest_prop_size":
            val = -1 if val is None else val
        else:
            val = val if val is not None else 0
        row[col] = val

    candidate_rows.append(row)
    candidate_names.append(smiski["name"])

candidate_matrix = pd.DataFrame(candidate_rows, columns=candidate_cols).values
predicted_vector = pd.DataFrame([raw_profile])[candidate_cols].values

distances = euclidean_distances(predicted_vector, candidate_matrix)[0]
closest_idx = np.argmin(distances)

print(f"\nPredicted morphology profile matched to: {candidate_names[closest_idx]}")
print(f"Distance: {distances[closest_idx]:.3f}")
print("\nAll candidates ranked:")
for idx in np.argsort(distances):
    print(f"  {candidate_names[idx]}: {distances[idx]:.3f}")


Predicted morphology profile matched to: SMISKI Doing Crunches
Distance: 1.000

All candidates ranked:
  SMISKI Doing Crunches: 1.000
  SMISKI Hoop: 1.732


In [ ]:
import joblib

model_bundle = {
    "model": multi_model,
    "feature_cols": feature_cols,
    "target_cols": target_cols,
    "encoders": encoders,
    "bool_cols": bool_cols,
}

joblib.dump(model_bundle, "/smiski_model.joblib")
print("Model saved to backend/app/ml/smiski_model.joblib")

Model saved to backend/app/ml/smiski_model.joblib
